In [1]:
import time
from pathlib import Path

import pandas as pd
from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import teams

In [2]:
SEASON = "2023-24"
SEASON_TYPE = "Regular Season"
RAW_OUTPUT = Path("../data/raw/shots.csv")
RAW_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

In [3]:
all_teams = teams.get_teams()
teams_df = pd.DataFrame(all_teams)
teams_df.head(10)

,id,full_name,abbreviation,nickname,city,state,year_founded
0,1610612737,Atlanta Hawks,ATL,Hawks,Atlanta,Georgia,1949
1,1610612738,Boston Celtics,BOS,Celtics,Boston,Massachusetts,1946
2,1610612739,Cleveland Cavaliers,CLE,Cavaliers,Cleveland,Ohio,1970
3,1610612740,New Orleans Pelicans,NOP,Pelicans,New Orleans,Louisiana,2002
4,1610612741,Chicago Bulls,CHI,Bulls,Chicago,Illinois,1966
5,1610612742,Dallas Mavericks,DAL,Mavericks,Dallas,Texas,1980
6,1610612743,Denver Nuggets,DEN,Nuggets,Denver,Colorado,1976
7,1610612744,Golden State Warriors,GSW,Warriors,San Francisco,California,1946
8,1610612745,Houston Rockets,HOU,Rockets,Houston,Texas,1967
9,1610612746,Los Angeles Clippers,LAC,Clippers,Los Angeles,California,1970


In [4]:
def fetch_team_shots(team_id: int, season: str, season_type: str) -> pd.DataFrame:
    shot_chart = shotchartdetail.ShotChartDetail(
        team_id=team_id,
        player_id=0,  # 0 = all players on that team
        season_nullable=season,
        season_type_all_star=season_type,
        context_measure_simple="FGA",
    )
    df = shot_chart.get_data_frames()[0]
    return df

In [5]:
all_shots = []

for _, row in teams_df.iterrows():
    team_id = row["id"]
    team_name = row["full_name"]
    
    try:
        df = fetch_team_shots(team_id, SEASON, SEASON_TYPE)
        all_shots.append(df)
        print(f"✓ {team_name}: {len(df)} shots")
    except Exception as e:
        print(f"✗ {team_name}: {e}")
    
    time.sleep(1.5)  # respect rate limit

shots_df = pd.concat(all_shots, ignore_index=True)
print(f"\nTotal shots collected: {len(shots_df):,}")

✓ Atlanta Hawks: 7584 shots
✓ Boston Celtics: 7396 shots
✓ Cleveland Cavaliers: 7147 shots
✓ New Orleans Pelicans: 7165 shots
✓ Chicago Bulls: 7339 shots
✓ Dallas Mavericks: 7352 shots
✓ Denver Nuggets: 7279 shots
✓ Golden State Warriors: 7515 shots
✓ Houston Rockets: 7459 shots
✓ Los Angeles Clippers: 7108 shots
✓ Los Angeles Lakers: 7177 shots
✓ Miami Heat: 7022 shots
✓ Milwaukee Bucks: 7258 shots
✓ Minnesota Timberwolves: 6974 shots
✓ Brooklyn Nets: 7307 shots
✓ New York Knicks: 7272 shots
✓ Orlando Magic: 6964 shots
✓ Indiana Pacers: 7599 shots
✓ Philadelphia 76ers: 7331 shots
✓ Phoenix Suns: 7063 shots
✓ Portland Trail Blazers: 7356 shots
✓ Sacramento Kings: 7455 shots
✓ San Antonio Spurs: 7436 shots
✓ Oklahoma City Thunder: 7324 shots
✓ Toronto Raptors: 7356 shots
✓ Utah Jazz: 7371 shots
✓ Memphis Grizzlies: 7229 shots
✓ Washington Wizards: 7493 shots
✓ Detroit Pistons: 7236 shots
✓ Charlotte Hornets: 7133 shots

Total shots collected: 218,700


In [6]:
print(shots_df.shape)
print(shots_df.columns.tolist())
shots_df.head()

(218700, 24)
['GRID_TYPE', 'GAME_ID', 'GAME_EVENT_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_NAME', 'PERIOD', 'MINUTES_REMAINING', 'SECONDS_REMAINING', 'EVENT_TYPE', 'ACTION_TYPE', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 'SHOT_ZONE_AREA', 'SHOT_ZONE_RANGE', 'SHOT_DISTANCE', 'LOC_X', 'LOC_Y', 'SHOT_ATTEMPTED_FLAG', 'SHOT_MADE_FLAG', 'GAME_DATE', 'HTM', 'VTM']


,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM
0,Shot Chart Detail,0022300018,12,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,10,56,...,Center(C),Less Than 8 ft.,6,-26,64,1,1,20231114,DET,ATL
1,Shot Chart Detail,0022300018,16,1630552,Jalen Johnson,1610612737,Atlanta Hawks,1,10,18,...,Left Side Center(LC),24+ ft.,26,-202,178,1,1,20231114,DET,ATL
2,Shot Chart Detail,0022300018,25,203992,Bogdan Bogdanović,1610612737,Atlanta Hawks,1,9,34,...,Left Side(L),8-16 ft.,15,-151,20,1,1,20231114,DET,ATL
3,Shot Chart Detail,0022300018,28,1629631,De'Andre Hunter,1610612737,Atlanta Hawks,1,9,2,...,Center(C),Less Than 8 ft.,3,-4,31,1,1,20231114,DET,ATL
4,Shot Chart Detail,0022300018,32,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,8,33,...,Center(C),Less Than 8 ft.,2,28,6,1,1,20231114,DET,ATL


In [7]:
shots_df.columns = shots_df.columns.str.lower()
shots_df.head()

,grid_type,game_id,game_event_id,player_id,player_name,team_id,team_name,period,minutes_remaining,seconds_remaining,...,shot_zone_area,shot_zone_range,shot_distance,loc_x,loc_y,shot_attempted_flag,shot_made_flag,game_date,htm,vtm
0,Shot Chart Detail,0022300018,12,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,10,56,...,Center(C),Less Than 8 ft.,6,-26,64,1,1,20231114,DET,ATL
1,Shot Chart Detail,0022300018,16,1630552,Jalen Johnson,1610612737,Atlanta Hawks,1,10,18,...,Left Side Center(LC),24+ ft.,26,-202,178,1,1,20231114,DET,ATL
2,Shot Chart Detail,0022300018,25,203992,Bogdan Bogdanović,1610612737,Atlanta Hawks,1,9,34,...,Left Side(L),8-16 ft.,15,-151,20,1,1,20231114,DET,ATL
3,Shot Chart Detail,0022300018,28,1629631,De'Andre Hunter,1610612737,Atlanta Hawks,1,9,2,...,Center(C),Less Than 8 ft.,3,-4,31,1,1,20231114,DET,ATL
4,Shot Chart Detail,0022300018,32,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,8,33,...,Center(C),Less Than 8 ft.,2,28,6,1,1,20231114,DET,ATL


In [8]:
shots_df.isnull().sum()[shots_df.isnull().sum() > 0]

Series([], dtype: int64)

In [9]:
shots_df["shot_made_flag"].value_counts(normalize=True).rename({1: "Made", 0: "Missed"})

shot_made_flag
Missed    0.525656
Made      0.474344
Name: proportion, dtype: float64

In [10]:
shots_df.to_csv(RAW_OUTPUT, index=False)
print(f"Saved {len(shots_df):,} rows to {RAW_OUTPUT}")

Saved 218,700 rows to ..\data\raw\shots.csv


In [11]:
verify = pd.read_csv(RAW_OUTPUT)
print(f"Reloaded shape: {verify.shape}")
verify.head()

Reloaded shape: (218700, 24)


,grid_type,game_id,game_event_id,player_id,player_name,team_id,team_name,period,minutes_remaining,seconds_remaining,...,shot_zone_area,shot_zone_range,shot_distance,loc_x,loc_y,shot_attempted_flag,shot_made_flag,game_date,htm,vtm
0,Shot Chart Detail,22300018,12,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,10,56,...,Center(C),Less Than 8 ft.,6,-26,64,1,1,20231114,DET,ATL
1,Shot Chart Detail,22300018,16,1630552,Jalen Johnson,1610612737,Atlanta Hawks,1,10,18,...,Left Side Center(LC),24+ ft.,26,-202,178,1,1,20231114,DET,ATL
2,Shot Chart Detail,22300018,25,203992,Bogdan Bogdanović,1610612737,Atlanta Hawks,1,9,34,...,Left Side(L),8-16 ft.,15,-151,20,1,1,20231114,DET,ATL
3,Shot Chart Detail,22300018,28,1629631,De'Andre Hunter,1610612737,Atlanta Hawks,1,9,2,...,Center(C),Less Than 8 ft.,3,-4,31,1,1,20231114,DET,ATL
4,Shot Chart Detail,22300018,32,1627749,Dejounte Murray,1610612737,Atlanta Hawks,1,8,33,...,Center(C),Less Than 8 ft.,2,28,6,1,1,20231114,DET,ATL
